# Tree Diagram Data Preprocessing
This notebook processes `sampled_genetic_disorders.csv` to produce `tree_data.csv`,
a summary table used by the D3 interactive tree diagram (Visualization 5).

The tree has four levels:
1. **Inheritance Side** — derived from `Genes in mother's side` and `Inherited from father`
2. **Gene Source** — derived from `Maternal gene` and `Paternal gene`
3. **Disorder Type** — from the `Genetic Disorder` column
4. **Disorder Subclass** — from the `Disorder Subclass` column (toggleable in the visualization)

Node sizes represent patient counts at each combination.

In [ ]:
import pandas as pd

df = pd.read_csv('sampled_genetic_disorders.csv')
print(f'Loaded {len(df)} rows')
df.head()

## Derive Inheritance Side
Combine `Genes in mother's side` and `Inherited from father` into a single category:
- **Both** — genes present on both sides
- **Maternal** — only on mother's side
- **Paternal** — only on father's side
- **Neither** — not flagged on either side

In [ ]:
def get_inheritance_side(row):
    maternal = row["Genes in mother's side"] == 'Yes'
    paternal = row['Inherited from father'] == 'Yes'
    if maternal and paternal:
        return 'Both'
    elif maternal:
        return 'Maternal'
    elif paternal:
        return 'Paternal'
    else:
        return 'Neither'

df['inheritance_side'] = df.apply(get_inheritance_side, axis=1)
print(df['inheritance_side'].value_counts())

## Derive Gene Source
Combine `Maternal gene` and `Paternal gene` into a single category:
- **Both** — gene passed from both mother and father
- **Mother** — passed from mother only
- **Father** — passed from father only
- **Neither** — not directly passed from either

In [ ]:
def get_gene_source(row):
    mother = row['Maternal gene'] == 'Yes'
    father = row['Paternal gene'] == 'Yes'
    if mother and father:
        return 'Both'
    elif mother:
        return 'Mother'
    elif father:
        return 'Father'
    else:
        return 'Neither'

df['gene_source'] = df.apply(get_gene_source, axis=1)
print(df['gene_source'].value_counts())

## Aggregate Patient Counts
Group by all four levels and count the number of patients in each combination.

In [ ]:
tree_data = (
    df.groupby(['inheritance_side', 'gene_source', 'Genetic Disorder', 'Disorder Subclass'])
      .size()
      .reset_index(name='count')
)

# Rename for clarity
tree_data = tree_data.rename(columns={'Genetic Disorder': 'disorder_type', 'Disorder Subclass': 'disorder_subclass'})

print(f'Output has {len(tree_data)} rows')
tree_data.head(10)

In [ ]:
# Quick sanity check — total should match original row count
print(f'Total patients in tree_data: {tree_data["count"].sum()}')
print(f'Total patients in original:  {len(df)}')

In [ ]:
# Save to CSV
tree_data.to_csv('tree_data.csv', index=False)
print('Saved tree_data.csv')
tree_data